# Pipelines
*Sequentially apply a list of transforms and a final estimator.*
- Scaling or imputation are examples of *transforms*
- a classifier is an *estimator*

In [23]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV

### The Mamography Mass dataset from UCI

In [26]:
mam_mass = pd.read_csv('MamMass.csv',na_values='?')

# mam_mass.shape
mam_mass.head(10)

,BI-RADS,Age,Shape,Margin,Density,Severity
0,5.0,67.0,3.0,5.0,3.0,1
1,4.0,43.0,1.0,1.0,NaN,1
2,5.0,58.0,4.0,5.0,3.0,1
3,4.0,28.0,1.0,1.0,3.0,0
4,5.0,74.0,1.0,5.0,NaN,1
5,4.0,65.0,1.0,NaN,3.0,0
6,4.0,70.0,NaN,NaN,3.0,0
7,5.0,42.0,1.0,NaN,3.0,0
8,5.0,57.0,1.0,5.0,3.0,1
9,5.0,60.0,NaN,5.0,1.0,1


In [ ]:
# Note: Given what 'Shape' and 'Margin' actually mean it is not really valid to treat 
# them as numeric/ordinal.
# Info on dataset available at https://archive.ics.uci.edu/ml/datasets/Mammographic+Mass
# i.e. Shape — the shape of the mass:

# 1 = round
# 2 = oval
# 3 = lobular
# 4 = irregular

# Margin — the margin/edge of the mass:

# 1 = circumscribed
# 2 = microlobulated
# 3 = obscured
# 4 = ill-defined
# 5 = spiculated

mam_mass.pop('BI-RADS')  # we're not using this variable in this example
y = mam_mass.pop('Severity').values
X = mam_mass.values

In [4]:
mam_mass.head()

,Age,Shape,Margin,Density
0,67.0,3.0,5.0,3.0
1,43.0,1.0,1.0,NaN
2,58.0,4.0,5.0,3.0
3,28.0,1.0,1.0,3.0
4,74.0,1.0,5.0,NaN


### Two sample missing value imputers from `sklearn`
- `SimpleImputer` replace missing values with the mean for that column
- `KNNImputer` use similar instances to estimate missing values

In [5]:
imp = SimpleImputer(missing_values=np.nan, strategy='mean') # Not used
imp_kNN = KNNImputer(missing_values = np.nan)
imp_kNN.fit(X)
Xi = imp_kNN.transform(X) # transform on the KNNImputed data

Also scale the data (otherwise `Age` attribute will dominate)

In [6]:
bScal = StandardScaler().fit(Xi)
XiS = bScal.transform(Xi) # X imputed and scaled = XiS

Making the train-test-split after Imputation and Scaling is **not** the right way to do things.

In [7]:
X_train, X_test, y_train, y_test = train_test_split(XiS, y, 
                                                    test_size=0.2,
                                                    random_state=42)
X_train.shape, X_test.shape

((768, 4), (193, 4))

In [8]:
knn = KNeighborsClassifier()
knn.fit(X_train,y_train)
y_pred = knn.predict(X_test)
print("Accuracy: {0:4.2f}".format(accuracy_score(y_test,y_pred)))
confusion_matrix(y_test, y_pred)


Accuracy: 0.84


array([[82, 19],
       [12, 80]], dtype=int64)

## Fit Impute and Scale transforms on Train data only
The right way to do it. 

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.2,
                                                    random_state=42)
X_train.shape, X_test.shape

((768, 4), (193, 4))

In [10]:
imp_kNN = KNNImputer(missing_values = np.nan)
imp_kNN.fit(X_train) # applying this just to X_train - fit always just on training
Xi_train = imp_kNN.transform(X_train) # transform always on both train and test
Xi_test = imp_kNN.transform(X_test)

In [11]:
bScal = StandardScaler().fit(Xi_train)
XiS_train = bScal.transform(Xi_train) # remember, this is X_train imputed and scaled
XiS_test = bScal.transform(Xi_test) # remember, this is X_test imputed and scaled

In [12]:
knn = KNeighborsClassifier()  #default hyperparameters 
knn.fit(XiS_train,y_train)
y_pred = knn.predict(XiS_test)
print("Accuracy: {0:4.2f}".format(accuracy_score(y_test,y_pred)))
confusion_matrix(y_test, y_pred)


Accuracy: 0.82


array([[78, 23],
       [12, 80]], dtype=int64)

In [13]:
knn.get_params()

{'algorithm': 'auto',
 'leaf_size': 30,
 'metric': 'minkowski',
 'metric_params': None,
 'n_jobs': None,
 'n_neighbors': 5,
 'p': 2,
 'weights': 'uniform'}

## With Pipelines - classic hold-out

In [14]:
kNNpipe  = Pipeline(steps=[
    ('imputer', KNNImputer(missing_values = np.nan)),
    ('scaler', StandardScaler()),
    ('classifier', KNeighborsClassifier())])


In [15]:
kNNpipe.fit(X_train, y_train)
y_pred = kNNpipe.predict(X_test)
print("Accuracy: {0:4.2f}".format(accuracy_score(y_test,y_pred)))
confusion_matrix(y_test, y_pred)


Accuracy: 0.82


array([[78, 23],
       [12, 80]], dtype=int64)

## Pipelines & Cross Validation

In [16]:
kNNpipe  = Pipeline(steps=[
    ('imputer', KNNImputer(missing_values = np.nan)),
    ('scaler', StandardScaler()),
    ('classifier', KNeighborsClassifier())])

# accuracy array
acc_arr = cross_val_score(kNNpipe, 
                          X, 
                          y, 
                          cv=5, 
                          n_jobs = -1)
# N-Jobs = Number of jobs to run in parallel. 
# Training the estimator and computing the score are parallelized over the cross-validation splits. 
# None means 1 unless in a joblib.parallel_backend context. -1 means using all processors.

print("Accuracy: {0:4.2f}".format(sum(acc_arr)/len(acc_arr)))
confusion_matrix(y_test, y_pred)


Accuracy: 0.77


array([[78, 23],
       [12, 80]], dtype=int64)

Accuracy estimate with pipeline and cross-validation is worse than with hold-out - why?  
Hold-out split is a *lucky* split - change `random_state` and repeat. 

more reliable.. when i changed the random_state to 5, in the hold out above, the accuracy was lower


What it proves:
If the model was genuinely 82% accurate, changing random_state shouldn't matter much — you'd get roughly the same result every time regardless of how the data was split.
But the fact that accuracy dropped when you changed it proves the 0.82 was dependent on which samples ended up in the test set, not on the model's true ability.

## Pipelines & Grid Search

The pipeline

In [17]:
kNNpipe  = Pipeline(steps=[
    ('imputer', KNNImputer(missing_values = np.nan)),
    ('scaler', StandardScaler()),
    ('classifier', KNeighborsClassifier())])

*k*-NN hyperparameters to be set

In [18]:
param_grid = {'classifier__n_neighbors':[1,3,5,10], 
              'classifier__metric':['manhattan','euclidean'],
              'classifier__weights':['uniform','distance']}

In [19]:
pipe_gs = GridSearchCV(kNNpipe,param_grid,cv=10, 
                      verbose = 1, n_jobs = -1)

In [20]:
pipe_gs = pipe_gs.fit(X_train, y_train)

Fitting 10 folds for each of 16 candidates, totalling 160 fits


In [21]:
pipe_gs.best_params_

{'classifier__metric': 'manhattan',
 'classifier__n_neighbors': 10,
 'classifier__weights': 'uniform'}

In [22]:
y_pred_gs = pipe_gs.predict(X_test)
print("Accuracy: {0:4.2f}".format(accuracy_score(y_test,y_pred_gs)))
confusion_matrix(y_test, y_pred_gs)

Accuracy: 0.81


array([[82, 19],
       [18, 74]], dtype=int64)

How do the best parameters compare with the default parameters above?